# 03. Multi-seed training: MobileNetV3-Large and ResNet-18, five seeds, 8 epochs
Each run saves its best checkpoint (by validation macro-F1), its validation logits and a metrics row to Google Drive the moment it finishes. If the session dies, run all again: finished runs are skipped. The test split is never read here.

In [ ]:
# Cell 1: environment, Drive, Kaggle login
from google.colab import drive
drive.mount("/content/drive")
!pip install --quiet --upgrade kaggle scikit-learn
import os, json, time, torch, numpy as np, pandas as pd
from google.colab import userdata
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")  # Colab Secrets, never in the notebook
!mkdir --parents ~/.kaggle && echo "$KAGGLE_API_TOKEN" > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token
RUNS_DIRECTORY = "/content/drive/MyDrive/cassacheck_runs"
os.makedirs(RUNS_DIRECTORY, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "Runtime is not a GPU: Runtime, Change runtime type, T4 GPU"
print("device:", device, "| runs directory:", RUNS_DIRECTORY)

Mounted at /content/drive
device: cuda | runs directory: /content/drive/MyDrive/cassacheck_runs


In [ ]:
# Cell 2: images and the published split
!kaggle competitions download --competition cassava-leaf-disease-classification --path /content/data
!unzip -q -o /content/data/cassava-leaf-disease-classification.zip "train_images/*" -d /content/data
!wget --quiet --output-document /content/split_v1_seed42.csv https://raw.githubusercontent.com/Justine-abc/cassavacheck/main/data/split_v1_seed42.csv
labels = pd.read_csv("/content/split_v1_seed42.csv")
image_directory = "/content/data/train_images"
train_frame = labels[labels.split == "train"].reset_index(drop=True)
validation_frame = labels[labels.split == "validation"].reset_index(drop=True)
print("train", len(train_frame), "validation", len(validation_frame), "| test rows present but never read:", int((labels.split == "test").sum()))

100% 5.76G/5.76G [05:39<00:00, 18.2MB/s]

train 14977 validation 3210 | test rows present but never read: 3210


In [ ]:
# Cell 3: dataset, transforms, model factory
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import torch.nn as nn
from sklearn.metrics import f1_score

class CassavaDataset(Dataset):
    def __init__(self, frame, transform): self.frame, self.transform = frame, transform
    def __len__(self): return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        return self.transform(Image.open(f"{image_directory}/{row.image_id}").convert("RGB")), int(row.label)

normalize = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
train_transform = transforms.Compose([transforms.RandomResizedCrop(224, scale=(0.7, 1.0)), transforms.RandomHorizontalFlip(),
                                      transforms.RandomVerticalFlip(), transforms.ColorJitter(0.2, 0.2, 0.2), transforms.ToTensor(), normalize])
eval_transform = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(), normalize])

def build_model(backbone: str) -> nn.Module:
    if backbone == "mobilenetv3_large":
        network = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
        network.classifier[3] = nn.Linear(network.classifier[3].in_features, 5)
    elif backbone == "resnet18":
        network = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        network.fc = nn.Linear(network.fc.in_features, 5)
    else:
        raise ValueError(backbone)
    return network.to(device)

class_counts = torch.tensor(train_frame["label"].value_counts().sort_index().values, dtype=torch.float)
class_weights = (class_counts.sum() / class_counts).to(device)
validation_loader = DataLoader(CassavaDataset(validation_frame, eval_transform), batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

def validation_logits(network):
    network.eval(); logits, targets = [], []
    with torch.no_grad():
        for images, batch_targets in validation_loader:
            logits.append(network(images.to(device)).cpu()); targets.append(batch_targets)
    return torch.cat(logits), torch.cat(targets)

In [7]:
# Cell 4: one run = one backbone, one seed. Saves checkpoint, validation logits and a metrics row to Drive; skipped if already done.
EPOCHS = 8
BACKBONES = ["mobilenetv3_large", "resnet18"]
SEEDS = [0, 1, 2, 3, 4]

def run_name(backbone, seed): return f"{backbone}_seed{seed}"

def train_one(backbone, seed):
    name = run_name(backbone, seed)
    done_marker = f"{RUNS_DIRECTORY}/{name}.done"
    if os.path.exists(done_marker):
        print(f"{name}: already finished, skipping"); return
    torch.manual_seed(seed); np.random.seed(seed)
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(CassavaDataset(train_frame, train_transform), batch_size=64, shuffle=True, num_workers=2, pin_memory=True, generator=generator)
    network = build_model(backbone)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(network.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * len(train_loader))
    best_macro_f1, history, started = -1.0, [], time.time()
    for epoch in range(1, EPOCHS + 1):
        network.train(); running_loss, seen = 0.0, 0
        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)
            optimizer.zero_grad(); loss = criterion(network(images), targets); loss.backward(); optimizer.step(); scheduler.step()
            running_loss += loss.item() * images.size(0); seen += images.size(0)
        logits, targets_all = validation_logits(network)
        predictions = logits.argmax(1)
        accuracy = float((predictions == targets_all).float().mean())
        macro_f1 = float(f1_score(targets_all, predictions, average="macro"))
        history.append({"epoch": epoch, "train_loss": running_loss / seen, "validation_accuracy": accuracy, "validation_macro_f1": macro_f1})
        print(f"{name} epoch {epoch}/{EPOCHS}: loss {running_loss / seen:.4f} | val acc {accuracy:.4f} | val macro-F1 {macro_f1:.4f}")
        if macro_f1 > best_macro_f1:
            best_macro_f1 = macro_f1
            torch.save(network.state_dict(), f"{RUNS_DIRECTORY}/{name}_best.pt")
            torch.save({"logits": logits, "targets": targets_all}, f"{RUNS_DIRECTORY}/{name}_validation_logits.pt")
    best = max(history, key=lambda h: h["validation_macro_f1"])
    row = {"backbone": backbone, "seed": seed, "epochs": EPOCHS, "best_epoch": best["epoch"], "validation_accuracy": best["validation_accuracy"],
           "validation_macro_f1": best["validation_macro_f1"], "minutes": round((time.time() - started) / 60, 1)}
    pd.DataFrame([row]).to_csv(f"{RUNS_DIRECTORY}/{name}_metrics.csv", index=False)
    pd.DataFrame(history).to_csv(f"{RUNS_DIRECTORY}/{name}_history.csv", index=False)
    open(done_marker, "w").write(json.dumps(row))
    print(f"{name}: finished in {row['minutes']} min, best macro-F1 {row['validation_macro_f1']:.4f} at epoch {row['best_epoch']}")

for backbone in BACKBONES:
    for seed in SEEDS:
        train_one(backbone, seed)

mobilenetv3_large_seed0 epoch 1/8: loss 0.8916 | val acc 0.7869 | val macro-F1 0.6840
mobilenetv3_large_seed0 epoch 2/8: loss 0.6766 | val acc 0.7757 | val macro-F1 0.6843
mobilenetv3_large_seed0 epoch 3/8: loss 0.5716 | val acc 0.7953 | val macro-F1 0.7053
mobilenetv3_large_seed0 epoch 4/8: loss 0.4787 | val acc 0.8364 | val macro-F1 0.7251
mobilenetv3_large_seed0 epoch 5/8: loss 0.3919 | val acc 0.8358 | val macro-F1 0.7355
mobilenetv3_large_seed0 epoch 6/8: loss 0.3351 | val acc 0.8352 | val macro-F1 0.7282
mobilenetv3_large_seed0 epoch 7/8: loss 0.2819 | val acc 0.8312 | val macro-F1 0.7291
mobilenetv3_large_seed0 epoch 8/8: loss 0.2676 | val acc 0.8343 | val macro-F1 0.7314
mobilenetv3_large_seed0: finished in 22.2 min, best macro-F1 0.7355 at epoch 5
mobilenetv3_large_seed1 epoch 1/8: loss 0.8944 | val acc 0.7570 | val macro-F1 0.6646
mobilenetv3_large_seed1 epoch 2/8: loss 0.6738 | val acc 0.7788 | val macro-F1 0.6854
mobilenetv3_large_seed1 epoch 3/8: loss 0.5660 | val acc 0.78

100%|██████████| 44.7M/44.7M [00:00<00:00, 172MB/s]


resnet18_seed0 epoch 1/8: loss 1.0239 | val acc 0.7386 | val macro-F1 0.6103
resnet18_seed0 epoch 2/8: loss 0.7996 | val acc 0.7118 | val macro-F1 0.6156
resnet18_seed0 epoch 3/8: loss 0.7244 | val acc 0.8174 | val macro-F1 0.7063
resnet18_seed0 epoch 4/8: loss 0.6409 | val acc 0.8386 | val macro-F1 0.7324
resnet18_seed0 epoch 5/8: loss 0.5668 | val acc 0.8093 | val macro-F1 0.7045
resnet18_seed0 epoch 6/8: loss 0.4873 | val acc 0.8224 | val macro-F1 0.7199
resnet18_seed0 epoch 7/8: loss 0.4170 | val acc 0.8308 | val macro-F1 0.7275
resnet18_seed0 epoch 8/8: loss 0.3829 | val acc 0.8470 | val macro-F1 0.7470
resnet18_seed0: finished in 22.2 min, best macro-F1 0.7470 at epoch 8
resnet18_seed1 epoch 1/8: loss 1.0166 | val acc 0.7202 | val macro-F1 0.6184
resnet18_seed1 epoch 2/8: loss 0.8059 | val acc 0.7779 | val macro-F1 0.6710
resnet18_seed1 epoch 3/8: loss 0.7136 | val acc 0.6579 | val macro-F1 0.5979
resnet18_seed1 epoch 4/8: loss 0.6402 | val acc 0.7754 | val macro-F1 0.6822
resnet

In [8]:
# Cell 5: summary table across runs (run this any time; it reads whatever has finished)
import glob
rows = [pd.read_csv(path) for path in sorted(glob.glob(f"{RUNS_DIRECTORY}/*_metrics.csv"))]
if rows:
    summary = pd.concat(rows, ignore_index=True)
    print(summary.to_string(index=False))
    print()
    print(summary.groupby("backbone")[["validation_accuracy", "validation_macro_f1"]].agg(["mean", "std", "count"]).round(4))
    summary.to_csv(f"{RUNS_DIRECTORY}/all_runs_summary.csv", index=False)
else:
    print("no finished runs yet")

         backbone  seed  epochs  best_epoch  validation_accuracy  validation_macro_f1  minutes
mobilenetv3_large     0       8           5             0.835826             0.735453     22.2
mobilenetv3_large     1       8           5             0.846729             0.743894     21.9
mobilenetv3_large     2       8           7             0.835826             0.733155     22.0
mobilenetv3_large     3       8           7             0.838941             0.742203     22.0
mobilenetv3_large     4       8           8             0.838941             0.743717     21.9
         resnet18     0       8           8             0.847040             0.746951     22.2
         resnet18     1       8           5             0.839252             0.733527     22.7
         resnet18     2       8           7             0.846417             0.744077     22.4
         resnet18     3       8           5             0.837383             0.732741     22.4
         resnet18     4       8           8       